# 2 · The three operators of HDC

In notebook 1 we stored the warehouse trial as a graph and asked it exact questions. Here we
build a second representation of the same nodes and relationship types, using
**hyperdimensional computing (HDC)**. Every item becomes a **hypervector**: a very long list
of random numbers. We then compute with just three operators:

| Operator | Notation | What it's for |
|---|---|---|
| binding | $\mathbf{a} \otimes \mathbf{b}$ | pairing a key with its value |
| bundling | $\mathbf{a} \oplus \mathbf{b}$ | collecting several items into one |
| permutation | $\rho(\mathbf{a})$ | marking order, such as an edge's direction |

None of this is learned. There's no training and no model, just random numbers and
arithmetic.

## Setup

We use [torchhd](https://github.com/hyperdimensional-computing/torchhd), an open source library
for HDC built on top of PyTorch, and call its functions directly so you see the real API. `warehouse_trial.py` holds
the `NODES` and `EDGES` from notebook 1, so we don't repeat them here. Import it before
`lancedb`, because it also hides LanceDB's first-write log line.

In [1]:
from warehouse_trial import NODES, EDGES  # warehouse_trial.py sits next to this notebook

import lancedb
import torch
import torchhd

## Defining a hypervector

A hypervector here has $D = 10{,}000$ entries, each $+1$ or $-1$ with equal chance. torchhd
calls this the **MAP** model, after its three operations: Multiply, Add, Permute. We fix the
random seed, so every run gives the same hypervectors.

In [2]:
torch.manual_seed(0)
D = 10_000

names = [n["id"] for rows in NODES.values() for n in rows] + list(EDGES)
hv = torchhd.random(len(names), D, vsa="MAP")
codebook = dict(zip(names, hv))

print(len(names), "hypervectors:", names)
print("first 10 entries of R04:", codebook["R04"][:10].tolist())

14 hypervectors: ['R17', 'R04', 'R11', 'aisle-7', 'aisle-8', 'aisle-9', 'pallet', 'forklift', 'oil', 'reroute', 'slow_down', 'VISITED', 'HAS_HAZARD', 'HANDLED_BY']
first 10 entries of R04: [-1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0]


## How alike are two hypervectors?

We compare hypervectors with **cosine similarity**:

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|} = \frac{1}{D} \sum_{i=1}^{D} a_i b_i \quad \text{(for } \pm 1 \text{ entries)}$$

It's $1$ for identical hypervectors, $0$ for unrelated ones and $-1$ for exact opposites.

## Random hypervectors are almost orthogonal

Each product $a_i b_i$ is $+1$ or $-1$ at random, so the sum mostly cancels out. Its typical
size is $\sqrt{D}$, which means

$$\cos(\mathbf{a}, \mathbf{b}) \approx 0 \pm \frac{1}{\sqrt{D}} = 0 \pm 0.01$$

This is the key fact behind HDC. Any two random hypervectors are nearly unrelated, so each
node gets its own clearly distinct hypervector without anyone designing it.

In [3]:
for a, b in [("R04", "R11"), ("R04", "aisle-8"), ("aisle-8", "aisle-9"), ("oil", "VISITED")]:
    print(f"cos({a}, {b}) = {torchhd.cosine_similarity(codebook[a], codebook[b]).item():+.3f}")

cos(R04, R11) = -0.003
cos(R04, aisle-8) = -0.013
cos(aisle-8, aisle-9) = -0.002
cos(oil, VISITED) = +0.013


In [4]:
sims = torchhd.cosine_similarity(hv, hv)
off_diagonal = sims[~torch.eye(len(names), dtype=torch.bool)]
print(f"largest |cos| between two different hypervectors: {off_diagonal.abs().max().item():.3f}")
assert off_diagonal.abs().max() < 5 / D**0.5

largest |cos| between two different hypervectors: 0.030


## Store the codebook in LanceDB

The table of name → hypervector is called the **codebook**. It's fixed, and shared by the
whole fleet. We store it in LanceDB with a `name`, a `kind` (the node label, or
`relationship`) and the hypervector itself as a fixed-size list of 10,000 floats.

In [5]:
kinds = {n["id"]: label for label, rows in NODES.items() for n in rows} | {r: "relationship" for r in EDGES}

db = lancedb.connect("warehouse.lancedb")
table = db.create_table(
    "hypervectors",
    [{"name": n, "kind": kinds[n], "vector": codebook[n].tolist()} for n in names],
    mode="overwrite",
)
table.schema

name: string
kind: string
vector: fixed_size_list<item: float>[10000]
  child 0, item: float

## Clean-up: from a noisy hypervector back to a name

The operators below produce **noisy** hypervectors: close to a codebook entry, but not
equal to it. LanceDB's vector search is what turns a noisy hypervector back into a name. We
ask for the nearest codebook entries by cosine. LanceDB reports a cosine *distance*,
$1 - \cos$, so we convert it back to a similarity score.

In [6]:
def cleanup(query, k=3):
    hits = table.search(query.numpy()).metric("cosine").limit(k).to_list()
    return [(h["name"], round(1 - h["_distance"], 3)) for h in hits]

To test it, flip a random 30% of the entries of `aisle-9`. That's a lot of damage, but the
damaged copy is still far closer to `aisle-9` than to anything else in the codebook. Flipping a fraction $p$ of the entries gives a cosine of exactly $1 - 2p$, here $0.4$,
while every other entry sits near $0$.

In [7]:
noisy = codebook["aisle-9"].clone()
noisy[torch.randperm(D)[: int(0.3 * D)]] *= -1
cleanup(noisy)

[('aisle-9', 0.4), ('R17', 0.019), ('slow_down', 0.014)]

## Binding ⊗

Binding is a way to **associate** a key with its value. It multiplies two hypervectors element-wise:

$$(\mathbf{a} \otimes \mathbf{b})_i = a_i \, b_i$$

The result is again a ±1 hypervector of the same length $D$, and it's **unlike both inputs**:
$\cos(\mathbf{a} \otimes \mathbf{b}, \mathbf{a}) \approx 0$.

In [8]:
ab = torchhd.bind(codebook["R04"], codebook["aisle-8"])
print(f"cos(R04 ⊗ aisle-8, R04)     = {torchhd.cosine_similarity(ab, codebook['R04']).item():+.3f}")
print(f"cos(R04 ⊗ aisle-8, aisle-8) = {torchhd.cosine_similarity(ab, codebook['aisle-8']).item():+.3f}")

cos(R04 ⊗ aisle-8, R04)     = -0.001
cos(R04 ⊗ aisle-8, aisle-8) = +0.017


### Binding reverses itself

For ±1 entries, $a_i \cdot a_i = 1$, so $\mathbf{a} \otimes \mathbf{a} = \mathbf{1}$, the
all-ones hypervector. Binding with $\mathbf{a}$ again therefore removes it:

$$\mathbf{a} \otimes (\mathbf{a} \otimes \mathbf{b}) = (\mathbf{a} \otimes \mathbf{a}) \otimes \mathbf{b} = \mathbf{b}$$

In MAP, unbinding is the same operation as binding.

In [9]:
recovered = torchhd.bind(codebook["R04"], ab)
torch.equal(recovered, codebook["aisle-8"])

True

### A key-value pair

This is what binding is for. Bind a key to a value, `VISITED ⊗ aisle-9`, and you get one
hypervector that looks like neither of them. Bind it with the key again and the value comes
back, and the codebook search names it.

In [10]:
pair = torchhd.bind(codebook["VISITED"], codebook["aisle-9"])
value = torchhd.bind(codebook["VISITED"], pair)

print(cleanup(value))
assert torchhd.cosine_similarity(value, codebook["aisle-9"]) > 0.99

[('aisle-9', 1.0), ('R04', 0.01), ('slow_down', 0.01)]


## Bundling ⊕

Bundling is like **superposition**, combining multiple hypervectors into one representation that has the same dimensionality. Bundling _adds_ hypervectors element-wise:

$$(\mathbf{a} \oplus \mathbf{b} \oplus \mathbf{c})_i = a_i + b_i + c_i$$

Where binding gives a result unlike its inputs, bundling gives a result **like all of its
inputs**. Each input contributes $D$ to the dot product, and the rest is near-zero noise.
The sum's length is about $\sqrt{3D}$, so

$$\cos(\mathbf{a} \oplus \mathbf{b} \oplus \mathbf{c},\ \mathbf{a}) \approx \frac{D}{\sqrt{D}\,\sqrt{3D}} = \frac{1}{\sqrt{3}} \approx 0.577$$

In [11]:
robots = torchhd.multiset(torch.stack([codebook[n] for n in ["R17", "R04", "R11"]]))

for n in ["R17", "R04", "R11", "aisle-8", "oil"]:
    print(f"cos(R17 ⊕ R04 ⊕ R11, {n:8}) = {torchhd.cosine_similarity(robots, codebook[n]).item():+.3f}")

cos(R17 ⊕ R04 ⊕ R11, R17     ) = +0.576
cos(R17 ⊕ R04 ⊕ R11, R04     ) = +0.580
cos(R17 ⊕ R04 ⊕ R11, R11     ) = +0.572
cos(R17 ⊕ R04 ⊕ R11, aisle-8 ) = -0.003
cos(R17 ⊕ R04 ⊕ R11, oil     ) = -0.002


### R04's record as one hypervector

R04 has two key-value pairs: it visited aisle-8 and it visited aisle-9. We bind each pair
and bundle the results into one record:

$$\mathbf{R04_{rec}} = \mathbf{VISITED} \otimes \mathbf{aisle8} \ \oplus\ \mathbf{VISITED} \otimes \mathbf{aisle9}$$

In [12]:
record = torchhd.multiset(torch.stack([
    torchhd.bind(codebook["VISITED"], codebook["aisle-8"]),
    torchhd.bind(codebook["VISITED"], codebook["aisle-9"]),
]))

### Asking the record a question

"Where did R04 go?" Binding's multiplicative operations distribute over bundling, so unbinding the record with
`VISITED` unbinds every pair inside it at once:

$$\mathbf{VISITED} \otimes \mathbf{R04_{rec}} = \mathbf{aisle8} \oplus \mathbf{aisle9}$$

Both aisles score about $1/\sqrt{2} \approx 0.71$, and aisle-7, which R04 never visited,
scores near 0.

In [13]:
answer = torchhd.bind(codebook["VISITED"], record)
scores = {n: round(torchhd.cosine_similarity(answer, codebook[n]).item(), 3) for n in ["aisle-7", "aisle-8", "aisle-9"]}
print(scores)
assert scores["aisle-8"] > 0.5 and scores["aisle-9"] > 0.5 and abs(scores["aisle-7"]) < 0.1

{'aisle-7': -0.019, 'aisle-8': 0.706, 'aisle-9': 0.706}


### Bundles have a capacity

Every item you bundle in lowers each member's score, as $1/\sqrt{k}$ for $k$ items. The
noise from non-members stays at about $1/\sqrt{D}$, so members stay easy to find until $k$
gets large. A bigger $D$ buys you room for more items.

In [14]:
for k in [1, 2, 4, 8, len(names) - 1]:
    bundle = torchhd.multiset(hv[:k])
    member = torchhd.cosine_similarity(bundle, hv[0]).item()
    outsider = torchhd.cosine_similarity(bundle, hv[-1]).item()
    print(f"k = {k:2}   member {member:.3f}   1/√k = {k**-0.5:.3f}   non-member {outsider:+.3f}")

k =  1   member 1.000   1/√k = 1.000   non-member +0.016
k =  2   member 0.709   1/√k = 0.707   non-member +0.013
k =  4   member 0.492   1/√k = 0.500   non-member +0.014
k =  8   member 0.360   1/√k = 0.354   non-member -0.005
k = 13   member 0.281   1/√k = 0.277   non-member +0.003


## Permutation ρ

There's a problem with binding: multiplication is commutative, so
$\mathbf{R04} \otimes \mathbf{aisle8} = \mathbf{aisle8} \otimes \mathbf{R04}$. If we encoded
an edge as source ⊗ target, the edge R04 → aisle-8 and the edge aisle-8 → R04 would be the
same hypervector, and the edge would lose its direction.

If you want to read about this aspect in more detail, see our [blog post on why permutation is needed in HDC](https://hyperdimensionalcomputing.ai/blog/permutation-in-hdc/).

In [15]:
ba = torchhd.bind(codebook["aisle-8"], codebook["R04"])
commutes = torchhd.cosine_similarity(ab, ba).item()
print(f"cos(R04 ⊗ aisle-8, aisle-8 ⊗ R04) = {commutes:.3f}")
assert commutes == 1.0

cos(R04 ⊗ aisle-8, aisle-8 ⊗ R04) = 1.000


### A cyclic shift

Permutation $\rho$ rotates a hypervector's entries by one position, and the last entry
wraps around to the front:

$$\rho(\mathbf{a}) = (a_D, a_1, a_2, \ldots, a_{D-1})$$

The shifted hypervector is **unlike the original**, because every entry now sits next to a
random neighbour: $\cos(\rho(\mathbf{a}), \mathbf{a}) \approx 0$.

In [16]:
shifted = torchhd.permute(codebook["aisle-8"], shifts=1)
print("aisle-8   :", codebook["aisle-8"][:6].tolist(), "...", codebook["aisle-8"][-1].item())
print("ρ(aisle-8):", shifted[:6].tolist())
print(f"cos(ρ(aisle-8), aisle-8) = {torchhd.cosine_similarity(shifted, codebook['aisle-8']).item():+.3f}")

aisle-8   : [-1.0, 1.0, 1.0, -1.0, 1.0, -1.0] ... -1.0
ρ(aisle-8): [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0]
cos(ρ(aisle-8), aisle-8) = -0.014


### Permutation is also reversible

Shifting back the other way, $\rho^{-1}$, gives the original exactly:
$\rho^{-1}(\rho(\mathbf{a})) = \mathbf{a}$. In torchhd, that's `shifts=-1`.

In [17]:
torch.equal(torchhd.permute(shifted, shifts=-1), codebook["aisle-8"])

True

### Permutation can model edge-direction

Binding alone can't tell source from target, as we saw above.
To solve this, permutation operates on the target _before_ binding, so that swapping source and target gives a different
hypervector:

$$\mathbf{R04} \otimes \rho(\mathbf{aisle8}) \ \ne\ \mathbf{aisle8} \otimes \rho(\mathbf{R04})$$

Now that we encoded the notion of order (source vs. target) in the bound representation, bundling them has no further consequences.

In [18]:
forward = torchhd.bind(codebook["R04"], torchhd.permute(codebook["aisle-8"], shifts=1))
backward = torchhd.bind(codebook["aisle-8"], torchhd.permute(codebook["R04"], shifts=1))
print(f"cos(R04 ⊗ ρ(aisle-8), aisle-8 ⊗ ρ(R04)) = {torchhd.cosine_similarity(forward, backward).item():+.3f}")

cos(R04 ⊗ ρ(aisle-8), aisle-8 ⊗ ρ(R04)) = -0.012


To read the target back out of `forward`, undo the steps in reverse order. Unbind the source,
then shift back:

$$\rho^{-1}(\mathbf{R04} \otimes (\mathbf{R04} \otimes \rho(\mathbf{aisle8}))) = \rho^{-1}(\rho(\mathbf{aisle8})) = \mathbf{aisle8}$$

In [19]:
target = torchhd.permute(torchhd.bind(codebook["R04"], forward), shifts=-1)
cleanup(target)

[('aisle-8', 1.0), ('oil', 0.03), ('slow_down', 0.013)]

## Wrap-up

| Operator | Math | What it does to similarity | Reversible? |
|---|---|---|---|
| binding ⊗ | $a_i \, b_i$ | result is **unlike** its inputs | yes: bind again with the same key |
| bundling ⊕ | $a_i + b_i$ | result is **like all** of its inputs, about $1/\sqrt{k}$ each | approximately: clean up against the codebook |
| permutation ρ | cyclic shift | result is **unlike** its input | yes: $\rho^{-1}$ |

Every result is still one hypervector of length $D$. In notebook 3 we put all three together.
Each edge $(s, r, t)$ becomes $\mathbf{s} \otimes \mathbf{r} \otimes \rho(\mathbf{t})$, and
bundling all nine edges gives **the whole warehouse trial graph as one hypervector**, which
we can then ask questions.